In [ ]:
import pandas as pd
import os
import re

In [63]:

def read_data():
    year = 0
    while year not in [2018, 2019, 2020]:
        year = int(input("Choose the year that you'd like to process (2018, 2019, 2020): "))
        if year not in [2018, 2019, 2020]:
            print("Invalid year. Please enter 2018, 2019, or 2020.")
        elif year == 2018:
            cargo_df = pd.read_csv('bronze//cargo_2018_cleaned.csv')
            header_df = pd.read_csv('bronze//shipment_2018_cleaned.csv')
            tariff_df = pd.read_csv('bronze//tariff_2018_cleaned.csv')
            shipper_df = pd.read_csv('bronze//shipper_2018_cleaned.csv')
            countries_df = pd.read_csv('bronze//countries_2018.csv')
        elif year == 2019:
            cargo_df = pd.read_csv('bronze//cargo_2019_cleaned.csv')
            header_df = pd.read_csv('bronze//shipment_2019_cleaned.csv')
            tariff_df = pd.read_csv('bronze//tariff_2019_cleaned.csv')
            shipper_df = pd.read_csv('bronze//shipper_2019_cleaned.csv')
            countries_df = pd.read_csv('bronze//countries_2019.csv')
        elif year == 2020:
            cargo_df = pd.read_csv('bronze//cargo_2020_cleaned.csv')
            header_df = pd.read_csv('bronze//shipment_2020_cleaned.csv')
            tariff_df = pd.read_csv('bronze//tariff_2020_cleaned.csv')
            shipper_df = pd.read_csv('bronze//shipper_2020_cleaned.csv')
            countries_df = pd.read_csv('bronze//countries_2020.csv')
    return header_df, shipper_df, cargo_df, tariff_df, countries_df, year

header_df, shipper_df, cargo_df, tariff_df, countries_df, year = read_data()


C:\Users\BL257EV\AppData\Local\Temp\ipykernel_29396\4256136357.py:16: DtypeWarning: Columns (0: harmonized_number) have mixed types. Specify dtype option on import or set low_memory=False.
  tariff_df = pd.read_csv('bronze//tariff_2019_cleaned.csv')


In [ ]:
# Define expected schema for each dataframe
expected_schemas = {
    'Shipment': {
        'identifier': 'int64',
        'carrier_code': 'str',
        'vessel_name': 'str',
        'vessel_country_code': 'str',
        'port_of_unlading': 'str',
        'estimated_arrival_date': 'str',
        'actual_arrival_date': 'str',
        'weight_combined': 'str'
    },
    'Shipper': {
        'identifier': 'int64',
        'shipper_party_name': 'str',
        'city': 'str',
        'country_code': 'str'
    },
    'Cargo': {
        'identifier': 'int64',
        'container_number': 'str',
        'description_sequence_number': 'float64',
        'piece_count': 'float64',
        'description_text': 'str'
    },
    'Tariff': {
        'identifier': 'int64',
        'container_number': 'str',
        'description_sequence_number': 'float64',
        'harmonized_number': 'float64',
        'harmonized_value': 'str',
        'harmonized_weight_combined': 'str'
    },
    'Countries': {
        'country_code': 'str'
    }
}

def check_dtypes(df, name, expected_schema):
    """Check if dataframe columns match expected dtypes."""
    print(f"\nDatatype Check for {name}:")
    dtype_issues = False
    
    for col, expected_dtype in expected_schema.items():
        if col not in df.columns:
            print(f"  ❌ Missing column: {col}")
            dtype_issues = True
        else:
            actual_dtype = str(df[col].dtype)
            if actual_dtype != expected_dtype:
                print(f"  ⚠️  {col}: expected {expected_dtype}, got {actual_dtype}")
                dtype_issues = True
    
    if not dtype_issues:
        print(f"  ✓ All dtypes match expected schema")
    return not dtype_issues

def validate_dataframes(df, name):
    """Validate a dataframe for nulls, duplicates, and dtypes."""
    print(f"\nValidating {name} DataFrame:")
    print(f"Shape: {df.shape}")

    # Check for nulls
    nulls = df.isnull().sum()
    total_nulls = nulls.sum()
    print(f"Total null values: {total_nulls}")
    if total_nulls > 0:
        print("Nulls per column:")
        print(nulls[nulls > 0])

    # Check for duplicates
    duplicates = df.duplicated().sum()
    print(f"Total duplicate rows: {duplicates}")

    # Check datatypes
    schema_ok = check_dtypes(df, name, expected_schemas[name])

    if total_nulls == 0 and duplicates == 0 and schema_ok:
        print(f"✓ {name} DataFrame is clean.")
    else:
        print(f"✗ {name} DataFrame has issues.")

def validate_countries(df):
    """Separate validation for countries dataframe."""
    print("\nValidating Countries DataFrame:")
    print(f"Shape: {df.shape}")

    # Check for nulls
    nulls = df.isnull().sum()
    total_nulls = nulls.sum()
    print(f"Total null values: {total_nulls}")
    if total_nulls > 0:
        print("Nulls per column:")
        print(nulls[nulls > 0])

    # Check for duplicates
    duplicates = df.duplicated().sum()
    print(f"Total duplicate rows: {duplicates}")

    # Additional: check for unique country codes
    unique_codes = df['country_code'].nunique()
    print(f"Unique country codes: {unique_codes}")

    # Check datatypes
    schema_ok = check_dtypes(df, 'Countries', expected_schemas['Countries'])

    if total_nulls == 0 and duplicates == 0 and schema_ok:
        print("✓ Countries DataFrame is clean.")
    else:
        print("✗ Countries DataFrame has issues.")
        print(f'there are {total_nulls} null values')

In [64]:
# Validate the loaded dataframes
validate_dataframes(header_df, "Shipment")
validate_dataframes(shipper_df, "Shipper")
validate_dataframes(cargo_df, "Cargo")
validate_dataframes(tariff_df, "Tariff")

validate_countries(countries_df)


Validating Shipment DataFrame:
Shape: (5000000, 8)
Total null values: 25
Nulls per column:
vessel_country_code    25
dtype: int64
Total duplicate rows: 0

Datatype Check for Shipment:
  ✓ All dtypes match expected schema
✗ Shipment DataFrame has issues.

Validating Shipper DataFrame:
Shape: (9858566, 4)
Total null values: 14661982
Nulls per column:
shipper_party_name          8
city                  7268431
country_code          7393543
dtype: int64
Total duplicate rows: 0

Datatype Check for Shipper:
  ✓ All dtypes match expected schema
✗ Shipper DataFrame has issues.

Validating Cargo DataFrame:
Shape: (10000000, 5)
Total null values: 3320130
Nulls per column:
container_number        180
piece_count         3319855
description_text         95
dtype: int64
Total duplicate rows: 67948

Datatype Check for Cargo:
  ⚠️  description_sequence_number: expected float64, got int64
✗ Cargo DataFrame has issues.

Validating Tariff DataFrame:
Shape: (10000000, 6)
Total null values: 4371234
Nulls

In [65]:
# Clean the dataframes: drop nulls in key columns and handle duplicates

def clean_shipment(df):
    # Key: identifier
    df_clean = df.dropna(subset=['identifier']).drop_duplicates()
    return df_clean

def clean_shipper(df):
    # Key: identifier
    df_clean = df.dropna(subset=['identifier']).drop_duplicates()
    return df_clean

def clean_cargo(df):
    # Key: (identifier, container_number, description_sequence_number)
    df_clean = df.dropna(subset=['identifier', 'container_number', 'description_sequence_number']).drop_duplicates()
    return df_clean

def normalize_weight_to_kg(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    match = re.match(r'^([0-9,.]+)\s*([A-Za-z]+)', text)
    if not match:
        return None
    amount = float(match.group(1).replace(',', ''))
    unit = match.group(2).lower()

    if unit in ['kilogram', 'kilograms', 'kg']:
        return amount
    if unit in ['pound', 'pounds', 'lb', 'lbs']:
        return amount * 0.45359237
    if unit in ['ton', 'tons', 't']:
        return amount * 1000.0
    return None

def clean_tariff(df):
    df_clean = df.dropna(
        subset=['identifier', 'container_number', 'description_sequence_number', 'harmonized_number', 'harmonized_weight_combined']
    ).drop_duplicates().copy()

    df_clean['harmonized_number'] = pd.to_numeric(df_clean['harmonized_number'], errors='coerce')
    df_clean['harmonized_weight_kg'] = df_clean['harmonized_weight_combined'].apply(normalize_weight_to_kg)
    df_clean['harmonized_weight_combined'] = df_clean['harmonized_weight_kg'].apply(
        lambda x: f"{x:.3f} Kilograms" if pd.notna(x) else None
    )
    return df_clean

def clean_countries(df):
    # Key: country_code
    df_clean = df.dropna(subset=['country_code']).drop_duplicates()
    return df_clean

# Apply cleaning
header_df_clean = clean_shipment(header_df)
shipper_df_clean = clean_shipper(shipper_df)
cargo_df_clean = clean_cargo(cargo_df)
tariff_df_clean = clean_tariff(tariff_df)
countries_df_clean = clean_countries(countries_df)

print("After cleaning:")
print(f"Shipment: {header_df_clean.shape}")
print(f"Shipper: {shipper_df_clean.shape}")
print(f"Cargo: {cargo_df_clean.shape}")
print(f"Tariff: {tariff_df_clean.shape}")
print(f"Countries: {countries_df_clean.shape}")

After cleaning:
Shipment: (5000000, 8)
Shipper: (9858566, 4)
Cargo: (9931872, 5)
Tariff: (4721539, 7)
Countries: (332, 1)


In [66]:
# Re-validate after cleaning
print("Re-validation after cleaning:")
validate_dataframes(header_df_clean, "Shipment")
validate_dataframes(shipper_df_clean, "Shipper")
validate_dataframes(cargo_df_clean, "Cargo")
validate_dataframes(tariff_df_clean, "Tariff")
validate_countries(countries_df_clean)

Re-validation after cleaning:

Validating Shipment DataFrame:
Shape: (5000000, 8)
Total null values: 25
Nulls per column:
vessel_country_code    25
dtype: int64
Total duplicate rows: 0

Datatype Check for Shipment:
  ✓ All dtypes match expected schema
✗ Shipment DataFrame has issues.

Validating Shipper DataFrame:
Shape: (9858566, 4)
Total null values: 14661982
Nulls per column:
shipper_party_name          8
city                  7268431
country_code          7393543
dtype: int64
Total duplicate rows: 0

Datatype Check for Shipper:
  ✓ All dtypes match expected schema
✗ Shipper DataFrame has issues.

Validating Cargo DataFrame:
Shape: (9931872, 5)
Total null values: 3251986
Nulls per column:
piece_count         3251891
description_text         95
dtype: int64
Total duplicate rows: 0

Datatype Check for Cargo:
  ⚠️  description_sequence_number: expected float64, got int64
✗ Cargo DataFrame has issues.

Validating Tariff DataFrame:
Shape: (4721539, 7)
Total null values: 148390
Nulls per 

In [67]:
# Save cleaned data to silver layer
os.makedirs('silver', exist_ok=True)

header_df_clean.to_csv(f'silver/shipment_{year}_silver.csv', index=False)
shipper_df_clean.to_csv(f'silver/shipper_{year}_silver.csv', index=False)
cargo_df_clean.to_csv(f'silver/cargo_{year}_silver.csv', index=False)
tariff_df_clean.to_csv(f'silver/tariff_{year}_silver.csv', index=False)
countries_df_clean.to_csv(f'silver/countries_{year}_silver.csv', index=False)

print("Cleaned data saved to silver/ directory.")

Cleaned data saved to silver/ directory.


In [ ]:
# Inspect harmonized_weight_combined patterns
sample = tariff_df['harmonized_weight_combined'].dropna().astype(str)
print(sample.head(50).tolist())
print('unique units count:', sample.str.extract(r'([A-Za-z]+)$')[0].value_counts().to_dict())
print('contains pounds or lbs:', sample.str.contains('pound|lb', case=False, na=False).sum())
print('contains kg:', sample.str.contains('kg|kilogram', case=False, na=False).sum())

In [ ]:
# Inspect cleaned tariff weight values after normalization
clean_sample = tariff_df_clean['harmonized_weight_combined'].dropna().astype(str)
print(clean_sample.head(50).tolist())
print('unique units after cleaning:', clean_sample.str.extract(r'([A-Za-z]+)$')[0].value_counts().to_dict())
print('contains pounds or lbs after cleaning:', clean_sample.str.contains('pound|lb', case=False, na=False).sum())
print('contains kg after cleaning:', clean_sample.str.contains('kg|kilogram', case=False, na=False).sum())

In [ ]:
tariff_df_clean.head(25)